<a href="https://colab.research.google.com/github/dr-mushtaq/Research-Work/blob/main/Text_Summarization_Clean_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CONTINUATION: Untitled68 + Nabila WFA with BERT Integration**

This notebook combines:
1. **Untitled68.ipynb**: T5 model fine-tuning with WFA-generated summaries
2. **Nabila_WFA_with_bert.ipynb**: BART model fine-tuning for comparison

**All original code is preserved. New BART training is added as Phase 2.**

## **Install Libraries**

In [ ]:
!pip install rouge-score --quiet
!pip install openpyxl --quiet

  Preparing metadata (setup.py) ... done


## **Imports & NLTK Downloads**

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from collections import defaultdict
import heapq, torch, warnings
from transformers import T5ForConditionalGeneration, T5Tokenizer
from transformers import BartForConditionalGeneration, BartTokenizer
from torch.optim import AdamW
from transformers.optimization import get_linear_schedule_with_warmup
from rouge_score import rouge_scorer

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

# **WFA Helper Functions**

In [ ]:
def word_frequency(text):
    word_freq = defaultdict(int)
    stopwords_set = set(stopwords.words('english'))
    for word in word_tokenize(text.lower()):
        if word.isalpha() and word not in stopwords_set:
            word_freq[word] += 1
    return word_freq

def summarize_text(text, num_sentences=2):
    sentence_scores = defaultdict(int)
    word_freq = word_frequency(text)
    for sentence in sent_tokenize(text):
        for word in word_tokenize(sentence.lower()):
            if word in word_freq:
                sentence_scores[sentence] += word_freq[word]
    top_sentences = heapq.nlargest(num_sentences, sentence_scores, key=sentence_scores.get)
    return ' '.join(top_sentences)

# **Generate & Save Extractive Summaries**

In [ ]:
df = pd.read_excel('/content/dataset.xlsx')
df = df.rename(columns={'Text1': 'source_textwfa'})[['source_textwfa']].dropna()
df['summary'] = df['source_textwfa'].apply(summarize_text)
df.to_excel('/content/summarized_data.xlsx', index=False)
print(f"Generated summaries for {len(df)} documents")

Generated summaries for 34 documents


## **Load Data & Train/Test Split**

In [ ]:
df2 = pd.read_excel('/content/summarized_data.xlsx')
df2 = df2.rename(columns={'summary': 'target_text', 'source_textwfa': 'source_text'}).dropna()
train_df, test_df = train_test_split(df2, test_size=0.3, random_state=42)
train_df['source_text'] = 'summarize: ' + train_df['source_text']
test_df['source_text']  = 'summarize: ' + test_df['source_text']

print(f"Training set: {len(train_df)} samples")
print(f"Test set: {len(test_df)} samples")

Training set: 23 samples
Test set: 11 samples


## **Load T5 Model & Tokenizer**

In [ ]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
model     = T5ForConditionalGeneration.from_pretrained('t5-base').to(device)
tokenizer = T5Tokenizer.from_pretrained('t5-base')

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/892M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## **ROUGE Helper Function**

In [ ]:
def compute_rouge(prediction: str, target: str) -> dict:
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'], use_stemmer=True
    )
    return scorer.score(prediction, target)

# **Phase 1: T5 Fine-Tuning (Constant LR)**

In [ ]:
warnings.filterwarnings('ignore')

NUM_EPOCHS = 5
optimizer  = AdamW(model.parameters(), lr=5e-5)

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0

    for _, row in train_df.iterrows():
        input_ids = tokenizer(
            row['source_text'],
            return_tensors='pt',
            max_length=512,
            truncation=True
        ).input_ids.to(device)

        labels = tokenizer(
            row['target_text'],
            return_tensors='pt',
            max_length=512,
            truncation=True
        ).input_ids.to(device)

        outputs    = model(input_ids=input_ids, labels=labels)
        loss       = outputs.loss
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

    avg_train_loss = train_loss / len(train_df)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  |  Training loss: {avg_train_loss:.5f}")

    # Evaluation
    model.eval()
    rouge_scores = []

    with torch.no_grad():
        for _, row in test_df.iterrows():
            input_ids = tokenizer(
                row['source_text'],
                return_tensors='pt',
                max_length=512,
                truncation=True
            ).input_ids.to(device)

            generated_ids = model.generate(
                input_ids=input_ids,
                max_length=512,
                num_beams=4,
                early_stopping=True
            )
            generated_text = tokenizer.decode(
                generated_ids[0], skip_special_tokens=True
            )
            rouge_scores.append(
                compute_rouge(generated_text, row['target_text'])
            )

    avg_rouge = {
        metric: np.mean([s[metric].fmeasure for s in rouge_scores])
        for metric in rouge_scores[0]
    }
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  |  ROUGE: {avg_rouge}")

Epoch 1/5  |  Training loss: 0.45935
Epoch 1/5  |  ROUGE: {'rouge1': np.float64(0.6255785695948564), 'rouge2': np.float64(0.51607267337111), 'rougeL': np.float64(0.49751656736066696)}
Epoch 2/5  |  Training loss: 0.25461
Epoch 2/5  |  ROUGE: {'rouge1': np.float64(0.6416661317346755), 'rouge2': np.float64(0.539896862904376), 'rougeL': np.float64(0.5258550063502778)}
Epoch 3/5  |  Training loss: 0.14119
Epoch 3/5  |  ROUGE: {'rouge1': np.float64(0.601599255107843), 'rouge2': np.float64(0.48199951594811774), 'rougeL': np.float64(0.5187573076081315)}
Epoch 4/5  |  Training loss: 0.06368
Epoch 4/5  |  ROUGE: {'rouge1': np.float64(0.6287157025794059), 'rouge2': np.float64(0.5192937231065176), 'rougeL': np.float64(0.5420271308925793)}
Epoch 5/5  |  Training loss: 0.03862
Epoch 5/5  |  ROUGE: {'rouge1': np.float64(0.6059697691685528), 'rouge2': np.float64(0.4900142908325076), 'rougeL': np.float64(0.5152418882663597)}


# **Phase 2: T5 Fine-Tuning (Linear LR Scheduler)**

In [ ]:
optimizer   = AdamW(model.parameters(), lr=5e-5)
total_steps = len(train_df) * NUM_EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

for epoch in range(NUM_EPOCHS):
    # Training
    model.train()
    train_loss = 0.0

    for _, row in train_df.iterrows():
        input_ids = tokenizer(
            row['source_text'],
            return_tensors='pt',
            max_length=512,
            truncation=True
        ).input_ids.to(device)

        labels = tokenizer(
            row['target_text'],
            return_tensors='pt',
            max_length=512,
            truncation=True
        ).input_ids.to(device)

        outputs    = model(input_ids=input_ids, labels=labels)
        loss       = outputs.loss
        train_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

    avg_train_loss = train_loss / len(train_df)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  |  Training loss: {avg_train_loss:.5f}")

    # Evaluation
    model.eval()
    rouge_scores = []

    with torch.no_grad():
        for _, row in test_df.iterrows():
            input_ids = tokenizer(
                row['source_text'],
                return_tensors='pt',
                max_length=512,
                truncation=True
            ).input_ids.to(device)

            generated_ids = model.generate(
                input_ids=input_ids,
                max_length=512,
                num_beams=4,
                early_stopping=True
            )
            generated_text = tokenizer.decode(
                generated_ids[0], skip_special_tokens=True
            )
            rouge_scores.append(
                compute_rouge(generated_text, row['target_text'])
            )

    avg_rouge = {
        metric: np.mean([s[metric].fmeasure for s in rouge_scores])
        for metric in rouge_scores[0]
    }
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  |  ROUGE: {avg_rouge}")

print("\nT5 Training complete.")

Epoch 1/5  |  Training loss: 0.03218
Epoch 1/5  |  ROUGE: {'rouge1': np.float64(0.6032601059571845), 'rouge2': np.float64(0.47876691609503386), 'rougeL': np.float64(0.5074677968162947)}
Epoch 2/5  |  Training loss: 0.02653
Epoch 2/5  |  ROUGE: {'rouge1': np.float64(0.55822007663628), 'rouge2': np.float64(0.421895351430272), 'rougeL': np.float64(0.46503277738166343)}
Epoch 3/5  |  Training loss: 0.01574
Epoch 3/5  |  ROUGE: {'rouge1': np.float64(0.5410772944322765), 'rouge2': np.float64(0.39703690045295253), 'rougeL': np.float64(0.4410200444065417)}
Epoch 4/5  |  Training loss: 0.01102
Epoch 4/5  |  ROUGE: {'rouge1': np.float64(0.5697247610808618), 'rouge2': np.float64(0.4351893225063034), 'rougeL': np.float64(0.47262379400395155)}
Epoch 5/5  |  Training loss: 0.00758
Epoch 5/5  |  ROUGE: {'rouge1': np.float64(0.5990948312346185), 'rouge2': np.float64(0.48064386796084885), 'rougeL': np.float64(0.5042838572513799)}

T5 Training complete.


# **Final Evaluation on Test Set (T5 Model)**

In [ ]:
model.eval()
rouge_scores = []
results = []

with torch.no_grad():
    for _, row in test_df.iterrows():
        input_ids = tokenizer(
            row['source_text'],
            return_tensors='pt',
            max_length=512,
            truncation=True
        ).input_ids.to(device)

        generated_ids = model.generate(
            input_ids=input_ids,
            max_length=512,
            num_beams=4,
            early_stopping=True
        )
        generated_text = tokenizer.decode(
            generated_ids[0], skip_special_tokens=True
        )

        score = compute_rouge(generated_text, row['target_text'])
        rouge_scores.append(score)

        results.append({
            'source_text'    : row['source_text'].replace('summarize: ', ''),
            'reference_summary': row['target_text'],
            'generated_summary': generated_text,
            'rouge1'         : round(score['rouge1'].fmeasure, 4),
            'rouge2'         : round(score['rouge2'].fmeasure, 4),
            'rougeL'         : round(score['rougeL'].fmeasure, 4),
        })

# Average ROUGE scores for T5
avg_rouge1 = np.mean([s['rouge1'].fmeasure for s in rouge_scores])
avg_rouge2 = np.mean([s['rouge2'].fmeasure for s in rouge_scores])
avg_rougeL = np.mean([s['rougeL'].fmeasure for s in rouge_scores])

print("=" * 50)
print("    T5 MODEL - FINAL EVALUATION RESULTS")
print("=" * 50)
print(f"  ROUGE-1  :  {avg_rouge1:.4f}")
print(f"  ROUGE-2  :  {avg_rouge2:.4f}")
print(f"  ROUGE-L  :  {avg_rougeL:.4f}")
print("=" * 50)

# Save T5 results to Excel
results_df = pd.DataFrame(results)
results_df.to_excel('/content/t5_final_evaluation_results.xlsx', index=False)
print(f"\nT5 results saved to /content/t5_final_evaluation_results.xlsx")

    T5 MODEL - FINAL EVALUATION RESULTS
  ROUGE-1  :  0.5991
  ROUGE-2  :  0.4806
  ROUGE-L  :  0.5043

T5 results saved to /content/t5_final_evaluation_results.xlsx


# **Phase 3: BART Model Integration (From Nabila_WFA_with_bert)**

Loading and fine-tuning BART for comparison with T5

In [ ]:
print("\n" + "="*80)
print("PHASE 3: STARTING BART FINE-TUNING")
print("="*80 + "\n")

print("Loading BART model and tokenizer...")
bart_model = BartForConditionalGeneration.from_pretrained('facebook/bart-large-cnn').to(device)
bart_tokenizer = BartTokenizer.from_pretrained('facebook/bart-large-cnn')
print("BART model loaded successfully.")


PHASE 3: STARTING BART FINE-TUNING

Loading BART model and tokenizer...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BART model loaded successfully.


## **BART Fine-Tuning Loop (3 Epochs with ROUGE evaluation)**

In [ ]:
print("\nStarting BART fine-tuning...\n")

bart_optimizer = AdamW(bart_model.parameters(), lr=5e-5)
NUM_EPOCHS_BART = 3

for epoch in range(NUM_EPOCHS_BART):

    # Training phase
    bart_model.train()
    train_loss = 0.0

    for index, row in train_df.iterrows():
        input_text = row['source_text']
        target_text = row['target_text']

        # Tokenize inputs and targets
        input_ids = bart_tokenizer(
            input_text,
            return_tensors='pt',
            max_length=512,
            truncation=True
        )['input_ids'].to(device)

        labels = bart_tokenizer(
            target_text,
            return_tensors='pt',
            max_length=512,
            truncation=True
        )['input_ids'].to(device)

        # Fine-tune the model
        bart_outputs = bart_model(input_ids=input_ids, labels=labels)
        loss = bart_outputs.loss
        train_loss += loss.item()

        loss.backward()
        bart_optimizer.step()
        bart_optimizer.zero_grad()

    avg_train_loss = train_loss / len(train_df)
    print(f"BART Epoch {epoch+1}/{NUM_EPOCHS_BART}  |  Training loss: {avg_train_loss:.5f}")

    # Evaluation phase
    bart_model.eval()
    bart_rouge_scores = []

    with torch.no_grad():
        for index, row in test_df.iterrows():
            input_text = row['source_text']
            target_text = row['target_text']

            # Tokenize input
            input_ids = bart_tokenizer(
                input_text,
                return_tensors='pt',
                max_length=512,
                truncation=True
            )['input_ids'].to(device)

            # Generate summaries
            generated_ids = bart_model.generate(
                input_ids=input_ids,
                max_length=512,
                num_beams=4,
                early_stopping=True
            )
            generated_text = bart_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

            # Compute ROUGE scores
            rouge_score = compute_rouge(generated_text, target_text)
            bart_rouge_scores.append(rouge_score)

    # Calculate average ROUGE scores for BART
    avg_bart_rouge = {
        metric: np.mean([score[metric].fmeasure for score in bart_rouge_scores])
        for metric in bart_rouge_scores[0]
    }
    print(f"BART Epoch {epoch+1}/{NUM_EPOCHS_BART}  |  ROUGE: {avg_bart_rouge}")

print("\nBART Training complete.")


Starting BART fine-tuning...

BART Epoch 1/3  |  Training loss: 0.22166
BART Epoch 1/3  |  ROUGE: {'rouge1': np.float64(0.689100999137548), 'rouge2': np.float64(0.6240949122614383), 'rougeL': np.float64(0.6326269900799661)}
BART Epoch 2/3  |  Training loss: 0.94903
BART Epoch 2/3  |  ROUGE: {'rouge1': np.float64(0.5179277377486424), 'rouge2': np.float64(0.43692953276606183), 'rougeL': np.float64(0.44265041259547383)}
BART Epoch 3/3  |  Training loss: 0.98435
BART Epoch 3/3  |  ROUGE: {'rouge1': np.float64(0.6336902835476214), 'rouge2': np.float64(0.5303455698895604), 'rougeL': np.float64(0.5083823746562737)}

BART Training complete.


## **Final Evaluation on Test Set (BART Model)**

In [ ]:
bart_model.eval()
bart_rouge_scores_final = []
bart_results = []

with torch.no_grad():
    for _, row in test_df.iterrows():
        input_ids = bart_tokenizer(
            row['source_text'],
            return_tensors='pt',
            max_length=512,
            truncation=True
        )['input_ids'].to(device)

        generated_ids = bart_model.generate(
            input_ids=input_ids,
            max_length=512,
            num_beams=4,
            early_stopping=True
        )
        generated_text = bart_tokenizer.decode(
            generated_ids[0], skip_special_tokens=True
        )

        score = compute_rouge(generated_text, row['target_text'])
        bart_rouge_scores_final.append(score)

        bart_results.append({
            'source_text'    : row['source_text'].replace('summarize: ', ''),
            'reference_summary': row['target_text'],
            'generated_summary': generated_text,
            'rouge1'         : round(score['rouge1'].fmeasure, 4),
            'rouge2'         : round(score['rouge2'].fmeasure, 4),
            'rougeL'         : round(score['rougeL'].fmeasure, 4),
        })

# Average ROUGE scores for BART
bart_avg_rouge1 = np.mean([s['rouge1'].fmeasure for s in bart_rouge_scores_final])
bart_avg_rouge2 = np.mean([s['rouge2'].fmeasure for s in bart_rouge_scores_final])
bart_avg_rougeL = np.mean([s['rougeL'].fmeasure for s in bart_rouge_scores_final])

print("\n" + "=" * 50)
print("    BART MODEL - FINAL EVALUATION RESULTS")
print("=" * 50)
print(f"  ROUGE-1  :  {bart_avg_rouge1:.4f}")
print(f"  ROUGE-2  :  {bart_avg_rouge2:.4f}")
print(f"  ROUGE-L  :  {bart_avg_rougeL:.4f}")
print("=" * 50)

# Save BART results to Excel
bart_results_df = pd.DataFrame(bart_results)
bart_results_df.to_excel('/content/bart_final_evaluation_results.xlsx', index=False)
print(f"\nBART results saved to /content/bart_final_evaluation_results.xlsx")


    BART MODEL - FINAL EVALUATION RESULTS
  ROUGE-1  :  0.6337
  ROUGE-2  :  0.5303
  ROUGE-L  :  0.5084

BART results saved to /content/bart_final_evaluation_results.xlsx


# **Comparison: T5 vs BART Model Performance**

In [ ]:
print("\n" + "=" * 70)
print("COMPARISON: T5 vs BART MODEL PERFORMANCE")
print("=" * 70)
print(f"\n{'Metric':<20} {'T5 Model':<20} {'BART Model':<20}")
print("-" * 70)
print(f"{'ROUGE-1':<20} {avg_rouge1:<20.4f} {bart_avg_rouge1:<20.4f}")
print(f"{'ROUGE-2':<20} {avg_rouge2:<20.4f} {bart_avg_rouge2:<20.4f}")
print(f"{'ROUGE-L':<20} {avg_rougeL:<20.4f} {bart_avg_rougeL:<20.4f}")
print("=" * 70)

# Determine which model performed better
print("\nModel Performance Summary:")
print("-" * 70)

t5_score = (avg_rouge1 + avg_rouge2 + avg_rougeL) / 3
bart_score = (bart_avg_rouge1 + bart_avg_rouge2 + bart_avg_rougeL) / 3

print(f"T5 Average ROUGE Score:     {t5_score:.4f}")
print(f"BART Average ROUGE Score:   {bart_score:.4f}")
print(f"\nBetter Model: {'BART' if bart_score > t5_score else 'T5'} " +
      f"(+{abs(bart_score - t5_score):.4f} difference)")
print("=" * 70)

print("\n✅ All training and evaluation complete!")
print("✅ Results saved to Excel files for further analysis")


COMPARISON: T5 vs BART MODEL PERFORMANCE

Metric               T5 Model             BART Model          
----------------------------------------------------------------------
ROUGE-1              0.5991               0.6337              
ROUGE-2              0.4806               0.5303              
ROUGE-L              0.5043               0.5084              

Model Performance Summary:
----------------------------------------------------------------------
T5 Average ROUGE Score:     0.5280
BART Average ROUGE Score:   0.5575

Better Model: BART (+0.0295 difference)

✅ All training and evaluation complete!
✅ Results saved to Excel files for further analysis
